In [9]:
import numpy as np
import pandas as pd
import os
import random
import tensorflow as tf


def set_seed(seed: int):
    random.seed(seed) # Python
    np.random.seed(seed)  # Numpy
    os.environ["PYTHONHASHSEED"] = str(seed)  # sistema operativo
    tf.random.set_seed(seed)  # TensorFlow

set_seed(25)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense
from tensorflow.keras import preprocessing

# Parameters
max_features = 20000  # Vocabulary size

# Load dataset
csv_path = '../datasets/human_or_ai_dataset_sub3.csv'
df = pd.read_csv(csv_path)

# Extract texts and labels
texts = df['text'].values
labels = df['source'].values

# Ensure numeric labels
label_map = {'human': 0, 'ai': 1}
y_data = np.array([label_map[label] for label in labels])

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=max_features)
x_data_tfidf = vectorizer.fit_transform(texts).toarray()
actual_features = x_data_tfidf.shape[1]
x_train = x_data_tfidf
y_train = y_data

In [11]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Bidirectional, Dropout, GlobalMaxPooling1D, Conv1D, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import initializers
from tensorflow.keras.layers import Flatten, Dense, Embedding, Input

# Model for TF-IDF features
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(actual_features,)))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

# Compile with better optimizer
model.compile(
    optimizer='adam',  # Adam typically works better than rmsprop
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


# Early stopping 
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

# Save best model
model_checkpoint = ModelCheckpoint(
    'best_model_sub3_s3.h5',
    monitor='val_accuracy',
    save_best_only=True
)

# Train with callbacks
history = model.fit(
    x_train, y_train,
    epochs=50,  # More epochs, early stopping will prevent overfitting
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping, model_checkpoint]
)

/opt/homebrew/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │     2,441,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,450,305 (9.35 MB)

 Trainable params: 2,450,305 (9.35 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
30/33 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6626 - loss: 0.6620

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6788 - loss: 0.6545 - val_accuracy: 0.9295 - val_loss: 0.4013
Epoch 2/50
30/33 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9642 - loss: 0.2779

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9648 - loss: 0.2688 - val_accuracy: 0.9524 - val_loss: 0.1492
Epoch 3/50
31/33 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9917 - loss: 0.0558

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9918 - loss: 0.0549 - val_accuracy: 0.9600 - val_loss: 0.1113
Epoch 4/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9997 - loss: 0.0153 - val_accuracy: 0.9543 - val_loss: 0.1091
Epoch 5/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9996 - loss: 0.0080 - val_accuracy: 0.9562 - val_loss: 0.1104
Epoch 6/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.0051 - val_accuracy: 0.9590 - val_loss: 0.1053
Epoch 7/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 0.0028 - val_accuracy: 0.9581 - val_loss: 0.1056
Epoch 8/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 0.0023 - val_accuracy: 0.9600 - val_loss: 0.1066


In [12]:
import pickle
from tensorflow.keras import preprocessing

competition_input = pd.read_csv('dataset2_disclosed_complete_inputs.csv', sep='\t')
print(f"Loaded competition input data with shape: {competition_input.shape}")
print(f"Columns: {competition_input.columns}")


# Separar os textos das labels
texts = competition_input['Text'].values
t_data = vectorizer.transform(texts).toarray()


Loaded competition input data with shape: (100, 2)
Columns: Index(['ID', 'Text'], dtype='object')


In [13]:
import tensorflow as tf
from tensorflow import keras  # Optional, but good for structured access
predictor = tf.keras.models.load_model('best_model_sub3_s3.h5')

In [14]:
# Make predictions
raw_predictions = predictor.predict(t_data)

# Convert probabilities to class labels (0 or 1)
predicted_labels = (raw_predictions > 0.5).astype(int).flatten()

# Map numerical predictions to text labels
label_map = {0: "Human", 1: "AI"}
predictions = [label_map[label] for label in predicted_labels]

# Create output dataframe
output_df = pd.DataFrame({
    'ID': competition_input['ID'],
    'Label': predictions
})


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 


In [15]:
# Optional: Verify against the provided dataset1_outputs.csv
try:
    ground_truth = pd.read_csv('dataset2_disclosed_complete_outputs.csv', sep='\t')
    merged = output_df.merge(ground_truth, on='ID', suffixes=('_pred', '_true'))
    accuracy = (merged['Label_pred'] == merged['Label_true']).mean()
    print(f"\nAccuracy on dataset1: {accuracy:.4f}")
    
    # Print confusion matrix
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(merged['Label_true'], merged['Label_pred'], labels=['Human', 'AI'])
    print("\nConfusion Matrix:")
    print("              Predicted")
    print("             Human    AI")
    print(f"True Human:  {cm[0][0]:5d}  {cm[0][1]:5d}")
    print(f"     AI:     {cm[1][0]:5d}  {cm[1][1]:5d}")
    
except Exception as e:
    print(f"Could not verify against ground truth: {e}")


Accuracy on dataset1: 0.7100

Confusion Matrix:
              Predicted
             Human    AI
True Human:     41     10
     AI:        18     30
